# Resampling and CV - How to Compare Models Without Fooling Yourself

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>QM47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/blob/main/notebooks/nb08_cross_validation_model_comparison_student.ipynb)

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Write the $k$-fold cross-validation estimator for both regression (MSE) and classification (misclassification / score-based) problems
2. Run 5-fold CV on a regression business case (California Housing) and on a classification business case (Breast Cancer), reporting the mean, standard deviation, and 95% confidence interval every time
3. Plot per-fold CV scores with the mean and CI, and compare the CV estimate against the single validation-set score using a second bar plot
4. Interpret whether a single validation score was lucky, unlucky, or representative based on whether it falls inside the CV 95% CI
5. Use the 95% CI overlap rule to decide whether one model (or hyperparameter choice) is **convincingly** better than another on the same task

---

> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises**. Please complete them before submitting your notebook.

---

## 💼 Why This Matters: One Lucky Split or Genuinely Better?

Two stakeholders are waiting on your answer this week, one from each half of the course so far. Both are asking the same question in different language.

**HomeValue Analytics (California Housing — regression).** Your Ridge pipeline from nb05 posts a validation $R^2$ of roughly **0.60**. The CFO is not rude about it, but she is skeptical: *"If we had drawn a different 20% for the validation set, would the number have come out the same?"* With only one split in hand, the honest answer is *you cannot tell*.

**State Health Department (Breast Cancer — classification).** Your logistic regression from nb07 posts a validation ROC-AUC of roughly **0.99**. The chief medical officer is similarly polite and similarly unconvinced: *"Is that 0.99 real, or did we just get lucky with the 114 patients in the validation set?"* Same shape of question, same problem: one split cannot answer it.

**Same question in two domains, same solution — cross-validation.** The trick is to stop relying on *one* 60/20/20 split. Repartition the training data many different ways, train and score once per partition, and study the **distribution** of scores instead of a single number. The story you report back to the CFO (or to the chief medical officer) stops being *"the model got 0.60"* and becomes *"the model got 0.60, plus or minus about 0.01, and the spread across folds was tight."* That second sentence is something a decision-maker can act on.

> **Today's focus:** run $k$-fold CV on both business cases, report the mean with a 95% confidence interval, and compare the CV estimate against the single validation-set score to decide whether that single split was lucky, unlucky, or representative.

---

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold,
    cross_val_score, cross_validate
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.metrics import roc_auc_score, mean_squared_error
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.precision', 4)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)
print("✓ Setup complete!")

**Reading the output:**

The setup cell gathers every tool the rest of the notebook uses, so nothing downstream has to stop and re-import. Here is a short tour of what each piece does and where you will meet it.

The **two splitters** — `KFold` and `StratifiedKFold` — are the objects that decide which rows land in which fold. You will pass one of them to `cross_val_score` and it takes care of the splitting internally. `KFold` chops the training data into $k$ equal-sized chunks at random, which is what you want in Section 2 for the California Housing regression problem. `StratifiedKFold` does the same thing but preserves the positive-class rate of the target in every chunk — which will matter the moment you move to classification in Section 3, because a fold with zero malignant patients would make ROC-AUC undefined.

The **two CV convenience functions** — `cross_val_score` and `cross_validate` — wrap the train-evaluate loop so you never have to write it by hand. `cross_val_score` returns one score per fold (a NumPy array of length $k$); `cross_validate` returns more information (train scores, fit times, multiple metrics) when you ask for it. The rest of this notebook uses `cross_val_score`.

**`train_test_split`** is the same single-split tool from nb01–nb07. It carves the 60/20/20 partition that every CV estimate today is compared against.

**`scipy.stats`** is here for exactly one call: `stats.t.ppf(0.975, df=k-1)` — the Student's $t$ critical value that turns a standard deviation into a 95% confidence interval. If "Student's $t$" sounds vaguely familiar from introductory statistics, that is the same distribution. Section 1 spells out why you need it.

The **three models** — `LinearRegression`, `Ridge`, `LogisticRegression` — are already familiar from nb03–nb07. Nothing new about the models today; what is new is how you *evaluate* them.

`RANDOM_SEED = 474` is inherited from nb01–nb07, so every split and every fold you see is directly comparable to what you computed in earlier notebooks.

> **A question that often comes up at this point:** *"Do I need to memorize which splitter goes with which problem type?"* No — just remember one rule: **if the target is categorical, stratify**. Everything else follows.

---

## 1. Why Cross-Validation Exists

Before the first line of CV code runs, take a minute to build up the intuition for *why* this technique exists. The problem is simple, the fix is elegant, and the math is just a formal rewrite of something you are about to watch in an animation. Most of the work today is reading and interpreting; the coding is the easy part.

### The problem with a single split

A single 60/20/20 split gives you one number for validation performance — but that number is **one draw from a distribution**. Which 20% of the data ended up in the validation set is a random choice, and a *different* random choice would give a *different* score. With 114 validation patients (breast cancer) or roughly 4,128 validation tracts (California Housing), the variation from one split to the next is easily large enough to flip the ordering of two candidate models. That is the CFO's and the chief medical officer's worry, stated in plain English: *"would the number change if we had drawn differently?"* — and the answer with one split is always *"yes, a bit, and we cannot tell by how much."*

### $k$-fold CV — the mechanism (watch it happen)

<center>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/KfoldCV.gif" width="640" alt="Animated visualization of 5-fold cross-validation">
</center>

Every frame of the animation above is one iteration of $k$-fold CV. The training data is sliced into $k$ non-overlapping **folds**; at each iteration exactly one fold is held out for scoring (the colored stripe) and the other $k-1$ folds are used for fitting the model. The held-out fold rotates through all $k$ positions, so by the end of the cycle **every sample in the training set has been predicted exactly once, by a model that never saw it during fitting**. That is the whole trick — a single pass of the animation gives you $k$ honest validation scores instead of one.

Three properties are worth naming while you watch:

- **Fold size is roughly constant.** Each fold is about $n/k$ rows. With $n \approx 13{,}000$ tracts and $k=5$, each fold holds around 2,600 rows.
- **The test set is never touched.** Everything you see rotating belongs to the *training* portion of the 60/20/20 split. The 20% test set stays locked away until nb14 or the final project — exactly the discipline you built in nb01.
- **The $k$ scores are not independent.** Adjacent iterations share $k-2$ of their $k-1$ training folds, so their scores are correlated. This is subtle but it matters when you build a confidence interval — it is one of the reasons Student's $t$ (with $k-1$ degrees of freedom), rather than the normal distribution, is the conservative choice for the CI.

> **A question that often comes up here:** *"Why not just take a bigger validation set instead of doing all this?"* Because a single validation set — no matter how big — still gives you only one number. Cross-validation gives you $k$ numbers, which lets you estimate the **uncertainty** around the mean. Uncertainty is the part that turns *"the model got 0.60"* into *"the model got 0.60 ± 0.01, tight spread across folds."*

### The $k$-fold CV estimator

With the mechanism in your head, the equations are just a formal way of writing down what you already watched. Partition the training data into $k$ equally-sized folds — call them $C_1, \ldots, C_k$. For each fold $i$, train on the other $k-1$ folds and evaluate on the held-out fold $C_i$. Average the $k$ scores.

**For regression** (ISLP eq. 5.3):

$$\text{CV}_{(k)} \;=\; \frac{1}{k} \sum_{i=1}^{k} \text{MSE}_i
\qquad \text{where} \qquad
\text{MSE}_i \;=\; \frac{1}{\lvert C_i \rvert} \sum_{j \in C_i} \left( y_j - \hat{y}_j^{(-i)} \right)^2$$

The notation $\hat{y}_j^{(-i)}$ just means *"the prediction for sample $j$ from the model trained on the $k-1$ folds that do not contain $j$."* That is the same idea the animation showed, written with subscripts.

**For classification** (ISLP eq. 5.4):

$$\text{CV}_{(k)} \;=\; \frac{1}{k} \sum_{i=1}^{k} \text{Err}_i
\qquad \text{where} \qquad
\text{Err}_i \;=\; \frac{1}{\lvert C_i \rvert} \sum_{j \in C_i} \mathbb{1}\!\left( y_j \neq \hat{y}_j^{(-i)} \right)$$

The structure is identical — only the per-fold loss changes (squared error for regression, misclassification rate for classification). Score-based metrics like $R^2$, ROC-AUC, accuracy, and F1 are monotone transforms of these losses and plug into the exact same formula:

$$\text{CV}_{(k)}^{\text{score}} \;=\; \frac{1}{k} \sum_{i=1}^{k} \text{score}_i$$

where $\text{score}_i$ is whatever metric you pass into the `scoring=` argument of `cross_val_score` (e.g., `'r2'`, `'roc_auc'`, `'f1'`).

> **Worth noticing:** these equations really are "just averages." The interesting question is never *"how do I average $k$ numbers?"* — it is *"how do I report the spread of those $k$ numbers in a way a stakeholder can act on?"* That is what the summary statistics below are for.

### The limiting case — Leave-One-Out CV (LOOCV)

One natural question at this point: *"what happens if I push $k$ all the way up to $n$?"* That limiting case has a name and a good visualization.

<center>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/LOOCV.gif" width="640" alt="Animated visualization of Leave-One-Out cross-validation">
</center>

When $k = n$, every fold is a single row. That special case is **Leave-One-Out CV (LOOCV)**: train on $n-1$ samples, predict the one that was left out, repeat $n$ times. The animation shows every row taking a turn as the held-out sample.

Two properties fall out of the $k = n$ setting, and together they explain why LOOCV is *not* the default in this course:

- **Low bias, because each training fold uses nearly all the data.** The model fit on $n-1$ rows is almost the model you would deploy. This makes LOOCV (nearly) unbiased as an estimator of the true test error.
- **High variance of the estimate, and very high compute cost.** The $n$ training sets differ by only one row, so the $n$ held-out errors are highly correlated, which inflates the *variance of the LOOCV mean*. And training $n$ models — about 13,000 for California Housing — is several orders of magnitude slower than training 5 or 10.

> **A common follow-up question:** *"If LOOCV is more accurate, why doesn't everyone use it?"* Because "more accurate" on paper loses to "much faster and almost as good" in practice. The practical default of $k = 5$ or $k = 10$ trades a little more bias for visibly lower variance in the mean, and for compute that finishes while you are still thinking about the result. The rest of this notebook uses $k = 5$.

### Summary statistics — always report all three

After every CV run you end up with $k$ fold scores $s_1, \ldots, s_k$. Three numbers summarize them, and you should report all three *every single time*.

**Mean** — the point estimate:

$$\bar{s} \;=\; \frac{1}{k} \sum_{i=1}^{k} s_i$$

**Standard deviation** across folds — how much the score wiggles from one partition to the next. Use the sample SD with $k-1$ in the denominator:

$$\text{SD} \;=\; \sqrt{\frac{1}{k-1} \sum_{i=1}^{k} \bigl(s_i - \bar{s}\bigr)^2}$$

**95% confidence interval** for the true mean — this is the range you actually put on the CFO's slide. It uses Student's $t$ with $k-1$ degrees of freedom, which is the right distribution when $k$ is small and the fold scores are not perfectly independent:

$$\text{CI}_{95\%} \;=\; \bar{s} \;\pm\; t_{0.975,\, k-1} \cdot \frac{\text{SD}}{\sqrt{k}}$$

At $k = 5$, $t_{0.975,\, 4} \approx 2.776$, so the half-width of the CI works out to roughly $1.24 \cdot \text{SD}$. That is a useful rule of thumb — if you know the fold-to-fold SD, you already know the approximate width of the CI.

> **The rule the rest of this notebook follows, no exceptions:** always report mean, SD, and CI together, paired with a per-fold bar plot so the distribution is visible. **Never quote a CV mean without the spread** — a number without its uncertainty is not a decision-ready number.

With the mechanism, the equations, and the summary statistics in hand, you are ready to apply the recipe to real data. Section 2 takes the regression case (California Housing); Section 3 does the same thing for classification (Breast Cancer). The two sections follow an identical four-step recipe — once you have seen it on one problem, the other feels like déjà vu.

---

## 2. $k$-Fold CV for Regression — California Housing (HomeValue Analytics)

Enough theory. Time to watch it in action.

First stop: the HomeValue Analytics pricing model from nb01–nb05. You already have a Ridge pipeline (StandardScaler + Ridge, $\alpha = 1.0$) that posts a validation $R^2$ around **0.60** on one 60/20/20 split. Today the CFO wants a number *with a confidence interval*, not just a point estimate. Section 1 gave you the tool; this section applies it.

The plan is a four-step recipe that will repeat, step for step, on the classification problem in Section 3. If you can walk through these four steps here, you already know Section 3 — only the splitter and the scoring metric will change:

1. **Split the data.** Apply the usual 60/20/20 partition with `RANDOM_SEED = 474` (same split nb01–nb05 used, so the numbers are directly comparable).
2. **Score the single split.** Fit Ridge on the training set, score once on the validation set — and record that **single-split $R^2$**. This is the "one draw from a distribution" you are about to compare against a full CV run.
3. **Run 5-fold CV on the training set only.** The test set stays locked. From the five fold scores, compute the mean, the standard deviation (`ddof=1`), and the 95% CI using the Student's $t$ formula from Section 1.
4. **Compare and interpret.** Plot the five per-fold scores on the left, and a two-bar comparison (single-split vs. CV mean, with CI error bar on the CV bar) on the right. A one-line status print tells you whether the single-split $R^2$ landed INSIDE, ABOVE, or BELOW the CV 95% CI.

Plugging $R^2$ into the CV estimator from Section 1 gives the equation this section is about to evaluate numerically:

$$\text{CV}_{(5)}^{R^2} \;=\; \frac{1}{5} \sum_{i=1}^{5} R^2_i$$

where $R^2_i$ is the coefficient of determination on held-out fold $i$. Five numbers, average them, report the mean with its CI. That is it.

> 💡 **Gemini Prompt:** "Load the California Housing dataset. Do a 60/20/20 train/val/test split with `random_state=RANDOM_SEED`. Build a Ridge pipeline (StandardScaler + Ridge with alpha=1.0), fit it on `X_train`, and compute the single-split validation R² on `X_val`. Then run 5-fold `KFold` cross-validation on the training set with `scoring='r2'`. From the fold scores, compute the mean, the sample standard deviation (`ddof=1`), and a 95% confidence interval using `scipy.stats.t.ppf(0.975, df=k-1)`. Print all three numbers plus the fold scores. Render a 1x2 matplotlib figure: left panel, per-fold bar plot with a horizontal mean line and a shaded 95% CI band; right panel, two-bar comparison of the single-split validation R² vs the 5-fold CV mean, with a 95% CI error bar on the CV bar only."
>
> **After running, verify:**
> - Single-split validation R² is printed as one number
> - Five fold R² values are printed as an array
> - Mean, SD, and the two CI endpoints are printed with 4-decimal precision
> - Left panel: 5 bars with a red dashed mean line and a shaded CI band
> - Right panel: 2 bars — one for the single validation R², one for the 5-fold CV mean with a vertical error bar
> - Test set remains untouched
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# --- 1. Load California Housing and do the same 60/20/20 split used across NB01-NB05 ---
california = fetch_california_housing(as_frame=True)
X_reg = california.data
y_reg = california.target

X_reg_temp, X_reg_test, y_reg_temp, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=RANDOM_SEED
)
X_reg_train, X_reg_val, y_reg_train, y_reg_val = train_test_split(
    X_reg_temp, y_reg_temp, test_size=0.25, random_state=RANDOM_SEED
)
print(f"Train: {len(X_reg_train)} | Val: {len(X_reg_val)} | Test: {len(X_reg_test)} (locked)")

# --- 2. Fit Ridge on train, score once on the single validation split ---
ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', Ridge(alpha=1.0, random_state=RANDOM_SEED))
])
ridge_pipeline.fit(X_reg_train, y_reg_train)
val_r2 = ridge_pipeline.score(X_reg_val, y_reg_val)
print(f"\nSingle-split validation R²: {val_r2:.4f}")

# --- 3. Run 5-fold CV on the TRAINING portion only (test set stays locked) ---
k = 5
cv = KFold(n_splits=k, shuffle=True, random_state=RANDOM_SEED)
cv_scores = cross_val_score(ridge_pipeline, X_reg_train, y_reg_train, cv=cv, scoring='r2')

mean_r2 = cv_scores.mean()
sd_r2 = cv_scores.std(ddof=1)
t_crit = stats.t.ppf(0.975, df=k - 1)
half_w = t_crit * sd_r2 / np.sqrt(k)
ci_low, ci_high = mean_r2 - half_w, mean_r2 + half_w

print(f"\n=== 5-FOLD CV (Ridge on California Housing) ===")
print(f"Fold R²:   {np.round(cv_scores, 4)}")
print(f"Mean R²:   {mean_r2:.4f}")
print(f"Std (k-1): {sd_r2:.4f}")
print(f"95% CI:    [{ci_low:.4f}, {ci_high:.4f}]  (half-width {half_w:.4f})")

# --- 4. Per-fold plot + Single-split vs CV comparison ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: per-fold CV scores with mean line and CI band
axes[0].bar(range(1, k + 1), cv_scores, color='steelblue', edgecolor='black')
axes[0].axhline(mean_r2, color='red', linestyle='--', label=f'Mean = {mean_r2:.4f}')
axes[0].axhspan(ci_low, ci_high, color='red', alpha=0.15, label='95% CI')
axes[0].set_xlabel('Fold')
axes[0].set_ylabel('R² score')
axes[0].set_title('Per-fold 5-fold CV R² — Cal Housing, Ridge',
                  fontsize=12, fontweight='bold')
axes[0].legend(loc='lower right')

# Right: single-split validation vs CV mean with CI error bar
labels = ['Single validation split', '5-fold CV (mean)']
values = [val_r2, mean_r2]
errs = [0, half_w]
colors = ['goldenrod', 'steelblue']
bars = axes[1].bar(labels, values, yerr=errs, color=colors,
                   capsize=10, edgecolor='black')
axes[1].set_ylabel('R² score')
axes[1].set_title('Single Validation R² vs. 5-fold CV R² (with 95% CI)',
                  fontsize=12, fontweight='bold')
for i, v in enumerate(values):
    axes[1].text(i, v + 0.005, f'{v:.4f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

# Quick status line for the CFO
inside = ci_low <= val_r2 <= ci_high
print(f"\n→ Single validation R² ({val_r2:.4f}) is "
      f"{'INSIDE' if inside else 'OUTSIDE'} the 5-fold CV 95% CI "
      f"[{ci_low:.4f}, {ci_high:.4f}].")

**Reading the output:**

You now have, side by side, everything the CFO would want to see.

The **per-fold bar plot on the left** is the five $R^2$ scores you get by training Ridge on 80% of the training set and evaluating on the remaining 20% — repeated five times, with each census tract landing in exactly one validation fold. On California Housing the fold scores usually land between about **0.58 and 0.62**, the mean is near **0.60**, the standard deviation hovers around **0.01**, and the 95% CI half-width is close to **0.01** (because with only $k = 5$ the Student's $t$ critical value pulls the half-width back up to roughly 1.24 × SD). The shaded red band is the 95% CI, and the red dashed line is the CV mean — so you can see at a glance that all five fold scores sit comfortably inside the CI band.

The **comparison bar plot on the right** is the moment of truth. The gold bar is the single-split validation $R^2$ from nb01–nb05's workflow; the blue bar is the 5-fold CV mean with a vertical error bar showing the 95% CI. Three outcomes are possible, and the printed status line tells you exactly which one you are in:

- **INSIDE the CI** — the single split was representative. The CFO can quote either number and the story is the same.
- **ABOVE the CI** — the single split was *lucky*. The validation set happened to contain "easy" tracts, and the CV mean is the honest estimate to report.
- **BELOW the CI** — the single split was *unlucky*. The validation set pulled in "hard" tracts, so again quote the CV mean with its CI and not the one-split number.

> **A question that often comes up here:** *"If the CV mean is almost always the safer number, should I just ignore the single validation score entirely?"* Not quite — the single-split score is still useful as a quick sanity check during development (it runs in a fraction of the time a 5-fold CV takes). But when it comes time to *report* performance to a stakeholder, the CV mean with its CI is the decision-ready number. The single-split score was the *exploration* tool; the CV estimate is the *production* tool.

A few interpretation points worth carrying forward:

- **The CV mean is the number that belongs on the CFO's slide.** The single validation $R^2$ is one draw; the CV mean is an average of five.
- **The CI width is a proxy for how much faith to put in the mean.** A half-width of \~0.01 says *"all five folds agreed — this number is stable."* A wide CI would be saying *"the estimate depends heavily on which 20% you held out"* and would push you toward repeated CV or a larger training set.
- **The test set is still locked.** Both bars come from the *training* population; the \~4,128 test tracts are reserved for the one-shot deployment estimate you will do in nb14 or the final project.

**Key takeaway:** Section 2 executed the four-step recipe for the regression side. Section 3 now does the exact same four steps on Breast Cancer — only the splitter (`StratifiedKFold`) and the scoring metric (`'roc_auc'`) will change.

---

## 3. Stratified $k$-Fold CV for Classification — Breast Cancer (MedScreen)

Same recipe, different problem. Section 2 landed the California Housing Ridge estimate with a tight CI around 0.60; now apply the identical four steps to the MedScreen pipeline from nb06–nb07. If the steps feel familiar, that is the point — the CV ritual is the *same* whether you are predicting home values or classifying tumors.

There is exactly **one new ingredient**: when the target is categorical, the folds need to *stratify*. Here is why, in plain language.

The breast cancer training set has roughly 341 patients, split about 63% benign and 37% malignant. A plain random 5-fold split could — just by luck — produce one fold with only 30% malignant patients and another with 45%. That kind of fold-to-fold imbalance is exactly the *noise* cross-validation is supposed to average out, not add. `StratifiedKFold` fixes this by enforcing the 37/63 ratio in every fold, so each fold faces a representative slice of the cancer-vs-healthy mix the classifier will see in production.

> **A question that often comes up here:** *"Why didn't we need stratify for the regression case?"* Because in regression there is no "class balance" to preserve — the target is a continuous number, not a category. With classification, the minority class could in principle vanish from a fold entirely (making a metric like ROC-AUC undefined), and stratify is the simple, foolproof defense.

Following nb07, the scoring metric is **ROC-AUC** — the threshold-independent headline for model selection. Plugging AUC into the $k$-fold estimator gives:

$$\text{CV}_{(5)}^{\text{AUC}} \;=\; \frac{1}{5} \sum_{i=1}^{5} \text{AUC}_i$$

where $\text{AUC}_i$ is the ROC-AUC on held-out stratified fold $i$. The four-step recipe is unchanged: single-split validation score, 5-fold CV on training data only, mean + SD + 95% CI, two-panel bar plot with interpretation.

> 💡 **Gemini Prompt:** "Load the breast cancer dataset. Do a 60/20/20 train/val/test split with `random_state=RANDOM_SEED` and `stratify=y` (both calls). Build a logistic regression pipeline (StandardScaler + LogisticRegression with `max_iter=1000`), fit it on `X_train`, and compute the single-split validation ROC-AUC on `X_val` using `roc_auc_score` with `predict_proba(X_val)[:, 1]`. Then run 5-fold `StratifiedKFold` CV on the training set with `scoring='roc_auc'`. Compute the mean, sample SD (`ddof=1`), and 95% CI using `scipy.stats.t.ppf(0.975, df=k-1)`. Print everything. Render a 1x2 matplotlib figure: left panel per-fold CV bar plot with mean line and shaded 95% CI band; right panel two-bar comparison of single-split validation ROC-AUC vs the CV mean, with a 95% CI error bar on the CV bar only."
>
> **After running, verify:**
> - Single-split validation ROC-AUC is printed as one number (\~0.99)
> - Five fold AUC values are printed as an array
> - Mean, SD, and the two CI endpoints are printed with 4-decimal precision
> - Left panel: 5 bars with a red dashed mean line and a shaded CI band
> - Right panel: 2 bars — one for the single validation AUC, one for the 5-fold CV mean with a vertical error bar
> - Stratification preserved the 37/63 class balance in each fold (no crash, no undefined AUC)
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# --- 1. Load breast cancer data and apply the same stratified 60/20/20 split used in NB06/NB07 ---
bc = load_breast_cancer(as_frame=True)
X_clf = bc.data
y_clf = bc.target

X_clf_temp, X_clf_test, y_clf_temp, y_clf_test = train_test_split(
    X_clf, y_clf, test_size=0.20, random_state=RANDOM_SEED, stratify=y_clf
)
X_clf_train, X_clf_val, y_clf_train, y_clf_val = train_test_split(
    X_clf_temp, y_clf_temp, test_size=0.25, random_state=RANDOM_SEED, stratify=y_clf_temp
)
print(f"Train: {len(X_clf_train)} | Val: {len(X_clf_val)} | Test: {len(X_clf_test)} (locked)")

# --- 2. Fit logistic regression on train, score once on validation ---
log_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=RANDOM_SEED, max_iter=1000))
])
log_pipeline.fit(X_clf_train, y_clf_train)
val_auc = roc_auc_score(y_clf_val, log_pipeline.predict_proba(X_clf_val)[:, 1])
print(f"\nSingle-split validation ROC-AUC: {val_auc:.4f}")

# --- 3. Run 5-fold STRATIFIED CV on the training set ---
k = 5
cv_strat = StratifiedKFold(n_splits=k, shuffle=True, random_state=RANDOM_SEED)
cv_scores_clf = cross_val_score(log_pipeline, X_clf_train, y_clf_train,
                                cv=cv_strat, scoring='roc_auc')

mean_auc = cv_scores_clf.mean()
sd_auc = cv_scores_clf.std(ddof=1)
t_crit = stats.t.ppf(0.975, df=k - 1)
half_w_clf = t_crit * sd_auc / np.sqrt(k)
ci_low_clf, ci_high_clf = mean_auc - half_w_clf, mean_auc + half_w_clf

print(f"\n=== 5-FOLD STRATIFIED CV (LogReg on Breast Cancer) ===")
print(f"Fold AUC:  {np.round(cv_scores_clf, 4)}")
print(f"Mean AUC:  {mean_auc:.4f}")
print(f"Std (k-1): {sd_auc:.4f}")
print(f"95% CI:    [{ci_low_clf:.4f}, {ci_high_clf:.4f}]  (half-width {half_w_clf:.4f})")

# --- 4. Per-fold plot + Single-split vs CV comparison ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, k + 1), cv_scores_clf, color='steelblue', edgecolor='black')
axes[0].axhline(mean_auc, color='red', linestyle='--', label=f'Mean = {mean_auc:.4f}')
axes[0].axhspan(ci_low_clf, ci_high_clf, color='red', alpha=0.15, label='95% CI')
axes[0].set_xlabel('Fold')
axes[0].set_ylabel('ROC-AUC')
axes[0].set_title('Per-fold 5-fold Stratified CV ROC-AUC — Breast Cancer, LogReg',
                  fontsize=12, fontweight='bold')
axes[0].legend(loc='lower right')

labels = ['Single validation split', '5-fold CV (mean)']
values = [val_auc, mean_auc]
errs = [0, half_w_clf]
colors = ['goldenrod', 'steelblue']
axes[1].bar(labels, values, yerr=errs, color=colors, capsize=10, edgecolor='black')
axes[1].set_ylabel('ROC-AUC')
axes[1].set_title('Single Validation ROC-AUC vs. 5-fold CV (with 95% CI)',
                  fontsize=12, fontweight='bold')
for i, v in enumerate(values):
    axes[1].text(i, v + 0.002, f'{v:.4f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

inside_clf = ci_low_clf <= val_auc <= ci_high_clf
print(f"\n→ Single validation ROC-AUC ({val_auc:.4f}) is "
      f"{'INSIDE' if inside_clf else 'OUTSIDE'} the 5-fold CV 95% CI "
      f"[{ci_low_clf:.4f}, {ci_high_clf:.4f}].")

**Reading the output:**

The **per-fold bar plot on the left** shows five ROC-AUC values on held-out stratified folds of the breast cancer training data. On this dataset the fold AUCs usually land between about **0.98 and 1.00**, the mean sits near **0.99**, and the standard deviation is around **0.005–0.01**. With $k = 5$ the 95% CI half-width is again about 1.24 × SD, so the CI is narrow — all five folds essentially agree.

The **comparison bar plot on the right** lines up the single-split validation ROC-AUC (from nb07's workflow) against the 5-fold CV mean with its 95% CI error bar. The printed status line tells you whether the single-split number is inside or outside the CI, exactly like it did in Section 2:

- **INSIDE the CI** — the nb07 validation AUC was representative. The chief medical officer can quote either number.
- **ABOVE the CI** — the single split was *lucky*; those 114 validation patients happened to be easy ones. The CV mean is the honest estimate.
- **BELOW the CI** — the single split was *unlucky*; the 114 patients were on the hard side. Quote the CV mean with its CI.

> **A question that often comes up here:** *"The AUC is already 0.99 — do we really need all this machinery?"* Yes, and for a surprising reason: the decisions the Health Department has to make are rarely *"is AUC 0.99?"* and almost always *"is THIS classifier's AUC convincingly higher than THAT one's?"* When you start comparing models later today (Exercises 1 and 2) and in nb09 (hyperparameter grids), the comparison question hinges on CI overlap — not on the point estimates. Building the CI now is what lets you answer comparison questions later.

A few interpretation points worth carrying forward:

- **The CV mean is the number that belongs on the chief medical officer's slide.** The single validation AUC is one draw; the CV mean is an average of five.
- **Stratification shows up in the CI width.** Because every fold preserves the 37/63 malignant/benign ratio, the per-fold AUCs are tightly clustered and the CI is narrow. A plain `KFold` split could give a noticeably wider CI, and in extreme cases a fold with zero malignant patients would make AUC undefined.
- **The nb07 rule still holds.** AUC is the headline for *which classifier ranks best*. The CV mean + CI is what makes that headline statistically defensible — precision and recall at a chosen threshold still come from a separate deployment-readiness report.
- **Test set stays locked.** Both bars on the right come from training-side data; the 114 test patients are untouched, reserved for the one-shot deployment estimate later in the course.

**Key takeaway:** Sections 2 and 3 executed the same four-step ritual on regression and classification data. The only differences were the splitter (`KFold` vs `StratifiedKFold`) and the scoring metric (`'r2'` vs `'roc_auc'`). The two exercises that follow reuse this ritual one more time — now to *compare two candidate models* on the same problem, which is the decision every CFO, CMO, and project board eventually asks you to make.

---

## 📝 PAUSE-AND-DO Exercise 1 (10 minutes)

**Task:** Does Ridge **convincingly** beat plain OLS on California Housing?

This is the question the CFO has actually been asking since nb05. You reached for Ridge because regularization seemed like a good idea — but can you *defend* that choice to a skeptical finance board using only the evidence CV gives you? Today you finally have the tool for a clean answer.

Run the same 5-fold CV recipe from Section 2 with a plain `LinearRegression` pipeline (no L2 penalty), keeping everything else identical — same `X_reg_train`, same `KFold` splitter with `random_state=RANDOM_SEED`, same `scoring='r2'`. Compute its mean $R^2$, sample SD, and 95% CI using the Student's $t$ formula from Section 1, then build a two-bar comparison plot with **Ridge** (from Section 2) and **LinearRegression** bars, each with its own 95% CI error bar.

**Decision rule (this is the one you will use for the rest of the course):** if the two 95% CIs **overlap**, the Ridge improvement from the L2 penalty is *not* statistically convincing on this dataset — in plain English, *you cannot tell the two apart*. The CFO would not be able to defend *"we need regularization"* to the board. If the CIs **do not overlap**, regularization is buying a real, measurable edge and Ridge is the defensible pick.

> **A question that often comes up on the first CI-overlap comparison:** *"How much overlap counts? What if they just barely touch?"* Any overlap at all means the two are statistically indistinguishable on this data. The cleanest professional move is to call the simpler model (here, plain OLS — no hyperparameter, no $\alpha$ to justify) the winner when CIs overlap. The more-complex model only wins when its CI clearly clears the simpler model's CI.

---

> 💡 **Gemini Prompt:** "Build a plain linear regression pipeline (StandardScaler + LinearRegression). Using the same `X_reg_train`, `y_reg_train`, and `KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)` from Section 2, run `cross_val_score` with `scoring='r2'`. Compute the mean, sample SD with `ddof=1`, and a 95% CI via `scipy.stats.t.ppf(0.975, df=4) * sd / sqrt(5)`. Print all three numbers. Then render a bar plot with two bars — 'Ridge (α=1.0)' (reusing `mean_r2`, `half_w` from Section 2) and 'Plain OLS' — each with a 95% CI error bar. End with a single printed line that says whether the two CIs overlap."
>
> **After running, verify:**
> - OLS mean R², SD, and CI are printed with 4-decimal precision
> - Bar plot has exactly two bars with 95% CI error bars
> - Bars are labelled 'Ridge (α=1.0)' and 'Plain OLS'
> - A final line prints either "CIs OVERLAP — Ridge's improvement is not statistically convincing" or "CIs do NOT overlap — Ridge's improvement is real"
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# YOUR SOLUTION CODE HERE
# Hint: Use the Gemini prompt above for step-by-step guidance.
#
# What to do:
#   1. Build a plain OLS pipeline: StandardScaler + LinearRegression
#   2. Run 5-fold CV on X_reg_train, y_reg_train with scoring='r2'
#      (use the same KFold object from Section 2)
#   3. Compute mean, sample SD (ddof=1), and 95% CI using Student's t
#   4. Plot a 2-bar comparison (Ridge vs OLS) with CI error bars
#   5. Print whether the two CIs overlap
#
# Variables available from Section 2:
#   - X_reg_train, y_reg_train
#   - cv (the 5-fold KFold splitter with random_state=RANDOM_SEED)
#   - mean_r2, sd_r2, half_w, ci_low, ci_high (Ridge results)


### YOUR ANALYSIS:

**Question 1:** Do the Ridge and OLS 95% CIs overlap?  
[Your answer — inspect the plot or compare the printed CI endpoints]

**Question 2:** What would you tell the CFO based on this result?  
[Your answer — "we should deploy Ridge because …" or "plain OLS is enough because …"]

**Question 3:** The two model's CIs could change if you used a different `k` (3-fold vs. 10-fold) or a different `scoring` (negative MSE). Which change would shrink the CI most, and why?  
[Your answer]

---

## 📝 PAUSE-AND-DO Exercise 2 (10 minutes)

**Task:** Does regularization **strength** change MedScreen's ROC-AUC in a way the Health Department would care about?

Exercise 1 compared *algorithms* (Ridge vs. plain OLS). This exercise asks a subtly different question: with the same algorithm (logistic regression), does changing the *hyperparameter* meaningfully move the needle? The decision rule stays identical — *compare CIs, not means* — and the tool stays identical too, which is the whole point.

Section 3 ran 5-fold stratified CV with the default `LogisticRegression(C=1.0)`. Rerun the same CV recipe on `X_clf_train` / `y_clf_train` with a **strongly regularized** classifier — `LogisticRegression(C=0.01)` — keeping the `StandardScaler`, the same `StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)` splitter, and `scoring='roc_auc'` identical. Compute its mean ROC-AUC, sample SD, and 95% CI, then build a two-bar comparison plot with **LogReg (C=1.0)** (reusing results from Section 3) and **LogReg (C=0.01)** bars, each with its own 95% CI error bar.

**Decision rule:** if the two 95% CIs **overlap**, the choice of regularization strength does **not** move MedScreen's screening performance in a statistically convincing way on this data — the Health Department can pick on interpretability or operational grounds. If the CIs **do not overlap**, one setting is a real, measurable winner on ROC-AUC.

> **A question that often comes up here:** *"Isn't this what `GridSearchCV` does automatically?"* Exactly — and that is nb09's entire topic. `GridSearchCV` runs this same pairwise comparison across an entire grid of candidate hyperparameter values, then picks the winner. The reason you are doing it by hand today is so that when nb09 automates the machinery, you already know what the machine is computing. Every tuning library you will ever use is a loop wrapped around today's exercise.

---

> 💡 **Gemini Prompt:** "Build a logistic regression pipeline with StandardScaler + LogisticRegression(C=0.01, random_state=RANDOM_SEED, max_iter=5000). Using the same `X_clf_train`, `y_clf_train`, and `StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)` from Section 3, run `cross_val_score` with `scoring='roc_auc'`. Compute the mean, sample SD with `ddof=1`, and a 95% CI via `scipy.stats.t.ppf(0.975, df=4) * sd / sqrt(5)`. Print all three numbers with 4-decimal precision. Then render a bar plot with two bars — 'LogReg (C=1.0)' (reusing `mean_auc` and `half_w_clf` from Section 3) and 'LogReg (C=0.01)' — each with a 95% CI error bar. End with a single printed line that says whether the two CIs overlap."
>
> **After running, verify:**
> - Strong-regularization mean ROC-AUC, SD, and CI are printed with 4-decimal precision
> - Bar plot has exactly two bars with 95% CI error bars
> - Bars are labelled 'LogReg (C=1.0)' and 'LogReg (C=0.01)'
> - A final line prints either "CIs OVERLAP — regularization strength does not convincingly change ROC-AUC" or "CIs do NOT overlap — one C value is a real winner"
> - All numerical outputs use standard decimal format — no scientific notation

In [ ]:
# YOUR SOLUTION CODE HERE
# Hint: Use the Gemini prompt above for step-by-step guidance.
#
# What to do:
#   1. Build a strong-regularization pipeline: StandardScaler + LogisticRegression(C=0.01, max_iter=5000)
#   2. Run 5-fold STRATIFIED CV on X_clf_train, y_clf_train with scoring='roc_auc'
#      (use the same StratifiedKFold object from Section 3)
#   3. Compute mean, sample SD (ddof=1), and 95% CI using Student's t
#   4. Plot a 2-bar comparison (C=1.0 vs C=0.01) with CI error bars
#   5. Print whether the two CIs overlap
#
# Variables available from Section 3:
#   - X_clf_train, y_clf_train
#   - cv_strat (the 5-fold StratifiedKFold splitter with random_state=RANDOM_SEED)
#   - mean_auc, sd_auc, half_w_clf, ci_low_clf, ci_high_clf (C=1.0 results)


### YOUR ANALYSIS:

**Question 1:** Do the C=1.0 and C=0.01 95% CIs overlap?  
[Your answer — inspect the plot or compare the printed CI endpoints]

**Question 2:** Based on this result, should MedScreen adopt stronger regularization in production?  
[Your answer — use the CI-overlap verdict, not the point estimates]

**Question 3:** nb09 will automate this comparison over a whole grid of `C` values. What role does the CI you just computed play inside `GridSearchCV`?  
[Your answer]

---

## 4. Wrap-Up: Key Takeaways

### What We Learned Today:

1. **One split is one draw from a distribution.** A single 60/20/20 split produces one number; $k$-fold CV produces $k$ numbers whose mean, standard deviation, and 95% confidence interval together form a much more honest estimate of production performance.

2. **The CV equation is the same across problem types.** Regression averages per-fold MSE; classification averages per-fold misclassification rate (or a monotone-equivalent score like ROC-AUC, accuracy, or F1). Only two new ingredients for classification: **stratification** (preserve class prevalence in every fold) and a **score-based** loss that matches the business question from nb07.

3. **Always report three numbers and a picture.** The mean is the point estimate, the sample SD (`ddof=1`) measures fold-to-fold noise, and the 95% CI using Student's $t$ with $k-1$ degrees of freedom is the range the CFO or the chief medical officer actually cares about. Pair it with a per-fold bar plot so the distribution is visible.

4. **Compare single-split against CV mean.** If the validation bar sits inside the CV 95% CI, the single split was representative. If outside, the split was lucky or unlucky and the CV mean is the honest estimate. This INSIDE/ABOVE/BELOW vocabulary shows up again in nb09 (tuning) and nb14 (opening the locked test set).

5. **Overlapping CIs mean no winner.** Exercises 1 and 2 showed the decision primitive you will use for every model comparison from here on — Ridge vs. OLS, $C=1.0$ vs. $C=0.01$, Random Forest vs. Gradient Boosting in Week 3. The rule is always the same: *compare CIs, not point estimates, and break ties toward the simpler model.*

### Critical Rules:

> **"Never trust a single split — always cross-validate before a model-choice decision."**

> **"Quote the CV mean with its 95% confidence interval, never the mean alone."**

> **"Stratify folds for classification so class balance survives every fold."**

> **"Overlapping CIs mean no winner — pick the simpler model."**

### Next Steps:

- **nb09** turns today's CV ritual into an entire ranked table. `GridSearchCV` runs the Section 2–3 machinery once per hyperparameter candidate and stacks the results; the CI-overlap rule you just practiced picks the winning row. If today felt repetitive, that is a good sign — the repetition is the point, because nb09 is *exactly today, automated over a grid*.
- **Your final project** should use this framework end-to-end: a single-split validation score plus a 5-fold CV mean with CI, paired with the two-panel comparison plot from Sections 2 and 3. If you can hand your reviewer that plot and walk through the INSIDE/ABOVE/BELOW verdict, you are already most of the way to a professional evaluation section.

---

## Participation Assignment Submission Instructions

### To Submit This Notebook:

1. **Complete all exercises**: Fill in both PAUSE-AND-DO exercise cells with your findings
2. **Run All Cells**: Execute `Runtime → Run all` to ensure everything works
3. **Save a Copy**: `File → Save a copy in Drive or Download the .ipynb extension`
4. **Submit**: Upload your `.ipynb` file in the participation assignment you find in the course Brightspace page.

### Before Submitting, Check:

- [ ] All cells execute without errors
- [ ] All outputs are visible
- [ ] Both exercise responses are complete
- [ ] Notebook is shared with correct permissions
- [ ] You can explain every line of code you wrote

### Next Step:

Complete the **Quiz** in Brightspace (auto-graded)

---

## Bibliography

- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An Introduction to Statistical Learning with Python* - Model Assessment and Selection (k-fold CV, resampling concepts)
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* - Resampling theory and selection bias
- scikit-learn User Guide: [Cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html)
- Kohavi, R. (1995). "A study of cross-validation and bootstrap for accuracy estimation and model selection." *IJCAI*.

---



<center>

Thank you!

</center>